In [ ]:
# Collection, scoring and aggregation live in ../pipeline.py; what makes this
# analysis different lives in config.py next to this notebook.
# Reads only the 0_raw snapshot, so this runs on a cluster with no scratch data.
# See ../../README.md for the snapshot and provenance model.
import sys
sys.path.insert(0, '..')   # pipeline.py
sys.path.insert(0, '.')    # config.py, if the kernel did not add it
import pipeline as pl
from config import CONFIG

# No seed is selected. Every (seed, fold) run is kept and reduced
#   5 folds -> mean -> 1 value per seed -> mean +- SD across 3 seeds
# writes per_run_results.csv, seed_level_results.csv,
#        representation_level_results.csv, coverage.csv, lomo_level_results.csv
per_run, seed_level, rep_level = pl.run(CONFIG)
rep_level

In [ ]:
# The seeds x folds grid should be complete. Any row here is a hole in it, and
# every number downstream of it rests on fewer runs than the Methods claim.
import pandas as pd
cov = pd.read_csv('coverage.csv')
print(f'{len(cov)} incomplete (model, group, seed) cells')
cov

In [ ]:
# One ROC-AUC per withheld molecule per model. `beta` is the beta chain: DeepNeo
# holds out beta chains where the others hold out alpha/beta pairs, so the beta is
# the only unit all five methods share and the only one a paired test can match on.
import pandas as pd
l = pd.read_csv('lomo_level_results.csv')
print(l.groupby('model').agg(molecules=('molecule', 'nunique'),
                             betas=('beta', 'nunique'),
                             median_auc=('roc_auc', 'median')).round(4).to_string())
l.head()